In [1]:
import pandas as pd
import time

In [2]:
df = pd.read_csv("list_210825.csv")
df

,S_no,CO_NAME,Tag
0,1,Kaarya Facilit.,non-f&o
1,2,RDB Real Estate,non-f&o
2,3,Dindigul Farm,non-f&o
3,4,Akiko,non-f&o
4,5,Nexxus Petro,non-f&o


In [3]:
Com_name_list = df.CO_NAME.to_list()
Com_name_list

['Kaarya Facilit.',
 'RDB Real Estate',
 'Dindigul Farm',
 'Akiko',
 'Nexxus Petro']

In [10]:
short_list = Com_name_list[0:5]
short_list

['Dynamic Cables', 'Yatra Online', 'NMDC Steel', 'Zaggle Prepaid']

In [4]:
from dotenv import load_dotenv
import os
from google import genai

load_dotenv()  # Loads variables from .env

api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash", contents="Explain how AI works in a few words"
)
print(response.text)

AI learns from data to recognize patterns and make intelligent decisions.


In [7]:
appending_list = {
    "CO_NAME": Com_name_list,
    "RISK": [""] * len(Com_name_list),
}

total_companies = len(Com_name_list)
batch_size = 5

for batch_num, start_idx in enumerate(range(0, total_companies, batch_size)):
    end_idx = min(start_idx + batch_size, total_companies)
    current_batch = Com_name_list[start_idx:end_idx]

    for idx, com in enumerate(current_batch):
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"""You are a corporate governance analyst. Using only the top 5 reputable public sources (news articles, press releases, and regulatory filings) from the past 12 months, produce a concise governance briefing for the public company {com}. Do not perform deep or exhaustive research — limit your search and citations to a maximum of five sources. Cite each source (headline + outlet + date + URL) next to the facts derived from it.

            Follow these steps and produce the output in the exact format specified below.

            Required steps for your analysis
            1. Search and select up to five high-quality, recent sources (within 12 months) about {com} — prioritize major business/financial outlets, company press releases, and regulatory filings (e.g., 8-K, proxy statement, regulator notices).
            2. From those sources, identify and summarize the top 5 corporate-governance-related events (see the checklist below). For each event, provide:
            - A one-line event title.
            - A 1 to 2 sentence factual summary (what happened, date).
            - Source citation(s) (headline + outlet + date + URL).
            3. Identify any key managerial resignations found in those sources (CEO, CFO, COO, CLO, Chair, or independent director). For each resignation provide:
            - Role and name.
            - Date of announcement and stated reason (if any).
            - Any unusual circumstances (e.g., suddenness, linked investigations, no succession plan).
            - Source citation.
            4. Provide a concise governance assessment (no more than 6 short bullets):
            - List observed red flags (explicitly label as “Red flags”).
            - List positive governance indicators (label as “Positive indicators”).
            - Give a final verdict sentence: one of (“Likely material governance concern,” “Minor governance concerns,” “No significant governance issues noted,” or “Inconclusive — more research needed”) with one short rationale (1 sentence).
            5. Methodology note (1 short paragraph): state the 5 sources used and confirm you limited yourself to up to five resources and 12 months.

            Checklist of event types to consider (use as guidance when selecting the top 5 events)
            - Board composition changes (appointments/resignations, committee chair changes, diversity shifts)
            - Executive compensation controversies or Say-on-Pay outcomes
            - Shareholder activism or proxy battles
            - Regulatory fines, investigations, or material weaknesses in controls
            - Major policy changes (auditor change, poison pill adoption, etc.)

            Required output format (strict)
            - Header line: Company: {com} — Governance Briefing (past 12 months)
            - Section A: Top 5 Governance Events — numbered 1 to 5, each with Title; 1to 2 sentence summary; citation(s)
            - Section B: Key Managerial Resignations — bullet list with details and citations
            - Section C: Governance Assessment — subheaders: Red flags; Positive indicators; Final verdict
            - Section D: Methodology — list the up to 5 sources used (headline + outlet + date + URL)

            Tone and constraints
            - Keep language factual, neutral, and concise. Total length should be no more than ~200 to 400 words.
            - Do not include opinions beyond the brief assessment and verdict.
            - Only use information that is directly supported by the cited sources.
            - Do not invent events, names, dates, or reasons.

            Example (format only — do not copy content)
            Company: Kaarya Facilit. — Governance Briefing (past 12 months)
            Section A: Top 5 Governance Events
            1. Board Chair resignation — On X date..., (Source: headline, outlet, date, URL)
            ...
            Section B: Key Managerial Resignations
            - CFO: Name — announced on X; reason; unusual circumstances. (Source)
            Section C: Governance Assessment
            Red flags:
            - ...
            Positive indicators:
            - ...
            Final verdict: Minor governance concerns — explanation.
            Section D: Methodology
            Sources used:
            1. Headline — Outlet — Date — URL
            2. ...

            Now produce the briefing for company {com}, using up to five sources from the past 12 months."""
        )
        appending_list["RISK"][start_idx + idx] = response.text
        time.sleep(5)  # sleep for 5 sec

    # Save the current progress as a DataFrame and CSV after each batch
    df_risk_batch = pd.DataFrame({
        "CO_NAME": Com_name_list[:end_idx],
        "RISK": appending_list["RISK"][:end_idx]
    })
    df_risk_batch.to_csv(f"risk_output_batch_{batch_num+1}.csv", index=False)
    display(df_risk_batch)  # Optional: show the DataFrame in notebook

,CO_NAME,RISK
0,Kaarya Facilit.,Company: Kaarya Facilit. — Governance Briefing...
1,RDB Real Estate,Company: RDB Real Estate — Governance Briefing...
2,Dindigul Farm,Company: Dindigul Farm — Governance Briefing (...
3,Akiko,Company: Akiko — Governance Briefing (past 12 ...
4,Nexxus Petro,Company: Nexxus Petro — Governance Briefing (p...
